<a href="https://colab.research.google.com/github/vaisanthragavc2025-pixel/vaisanth-EDA/blob/main/25BAI0145_EXP4_28.07.2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer

# Create sample DataFrame with missing values (NaN)
df = pd.DataFrame(
    {
        "Customer_Age": [34, np.nan, 28, 52, 41],
        "Purchase_Amount": [120.50, 250.00, np.nan, 410.00, 85.00],
        "Membership_Tier": ["Gold", "Silver", "Gold", np.nan, "Silver"],
    }
)

# 1. Mean Imputation (Numerical)
mean_imputer = SimpleImputer(strategy="mean")
df["Age_Mean"] = mean_imputer.fit_transform(df[["Customer_Age"]])

# 2. Median Imputation (Numerical - robust to outliers)
median_imputer = SimpleImputer(strategy="median")
df["Purchase_Median"] = median_imputer.fit_transform(df[["Purchase_Amount"]])

# 3. Mode Imputation (Categorical - most frequent)
mode_imputer = SimpleImputer(strategy="most_frequent")
df["Tier_Mode"] = mode_imputer.fit_transform(df[["Membership_Tier"]]).ravel()

# 4. MICE Imputation (Multivariate Imputation by Chained Equations)
# Select numerical columns for MICE
num_cols = ["Customer_Age", "Purchase_Amount"]
mice_imputer = IterativeImputer(max_iter=10, random_state=42)
df_mice = df[num_cols].copy()
df_mice.iloc[:, :] = mice_imputer.fit_transform(df_mice)
df["Age_MICE"] = df_mice["Customer_Age"]
df["Purchase_MICE"] = df_mice["Purchase_Amount"]

print(df)


   Customer_Age  Purchase_Amount Membership_Tier  Age_Mean  Purchase_Median  \
0          34.0            120.5            Gold     34.00           120.50   
1           NaN            250.0          Silver     38.75           250.00   
2          28.0              NaN            Gold     28.00           185.25   
3          52.0            410.0             NaN     52.00           410.00   
4          41.0             85.0          Silver     41.00            85.00   

  Tier_Mode   Age_MICE  Purchase_MICE  
0      Gold  34.000000     120.500000  
1    Silver  43.653153     250.000000  
2      Gold  28.000000     -14.034386  
3      Gold  52.000000     410.000000  
4    Silver  41.000000      85.000000  


In [ ]:
import numpy as np
import pandas as pd

# =====================================================================
# 3. Handling Missing Data (Setup & Identification)
# =====================================================================

# Initialize a base matrix of inventory stock levels
inventory_data = np.arange(15, 30).reshape(5, 3)

# Create DataFrame with Device Types as index and Warehouses as columns
df_inventory = pd.DataFrame(
    inventory_data,
    index=["laptops", "monitors", "servers", "routers", "switches"],
    columns=["wh_east", "wh_west", "wh_north"],
)

# Introduce missing values (NaN) to replicate real-world data gaps
df_inventory["wh_south"] = np.nan
df_inventory.loc["firewalls"] = np.arange(15, 19)
df_inventory.loc["access_points"] = np.nan
df_inventory["wh_central"] = np.nan

# Manually assign specific stock data to a single cell
# Note: Using .loc to avoid slice-assignment warnings
df_inventory.loc["laptops", "wh_south"] = 20.0

print("--- Original Inventory DataFrame with Missing Data ---")
print(df_inventory)
print("\n")

# 1. Check for missing values (returns boolean DataFrame)
print("--- 1. isnull() ---")
print(df_inventory.isnull())
print("\n")

# 2. Check for reported values (returns boolean DataFrame)
print("--- 2. notnull() ---")
print(df_inventory.notnull())
print("\n")

# 3. Count the number of missing entries per warehouse column
print("--- 3. Missing values per warehouse column ---")
print(df_inventory.isnull().sum())
print("\n")

# 4. Find the grand total of missing data points across the entire system
print("--- 4. Total missing data points ---")
print(df_inventory.isnull().sum().sum())
print("\n")

# 5. Count actual reported stock entries per warehouse
print("--- 5. Count of valid reported stock values ---")
print(df_inventory.count())
print("\n")

--- Original Inventory DataFrame with Missing Data ---
               wh_east  wh_west  wh_north  wh_south  wh_central
laptops           15.0     16.0      17.0      20.0         NaN
monitors          18.0     19.0      20.0       NaN         NaN
servers           21.0     22.0      23.0       NaN         NaN
routers           24.0     25.0      26.0       NaN         NaN
switches          27.0     28.0      29.0       NaN         NaN
firewalls         15.0     16.0      17.0      18.0         NaN
access_points      NaN      NaN       NaN       NaN         NaN


--- 1. isnull() ---
               wh_east  wh_west  wh_north  wh_south  wh_central
laptops          False    False     False     False        True
monitors         False    False     False      True        True
servers          False    False     False      True        True
routers          False    False     False      True        True
switches         False    False     False      True        True
firewalls        False    F

In [ ]:

# =====================================================================
# 4. Dropping Missing Data
# =====================================================================

# Filter a series using boolean indexing to see only valid data
print("--- Filtered Series: wh_south with data ---")
print(df_inventory.wh_south[df_inventory.wh_south.notnull()])
print("\n")

# Drop rows where 'wh_south' has a missing value
print("--- Drop rows where wh_south is NaN ---")
print(df_inventory.wh_south.dropna())
print("\n")

# Drop rows if ANY column contains a NaN value
print("--- Drop any row containing at least one NaN ---")
print(df_inventory.dropna())
print("\n")

# Drop only rows where every single column is entirely NaN
print("--- Drop rows that are completely empty (how='all') ---")
print(df_inventory.dropna(how="all"))
print("\n")

# Drop columns that are entirely NaN
print("--- Drop columns that are completely empty (axis=1) ---")
print(df_inventory.dropna(how="all", axis=1))
print("\n")

# Keep columns that have at least 5 non-null data entries
print("--- Keep columns with a minimum of 5 valid values (thresh=5) ---")
print(df_inventory.dropna(thresh=5, axis=1))
print("\n")


# =====================================================================
# Mathematical Operations with NaN
# =====================================================================

print("--- NumPy vs Pandas Math Comparison ---")
sample_array = np.array([100, 200, np.nan, 300])
sample_series = pd.Series(sample_array)

# NumPy returns nan if any item is nan; Pandas ignores nan by default
print(f"NumPy Mean: {sample_array.mean()} | Pandas Mean: {sample_series.mean()}")
print("\n")

print("--- Metrics for wh_south column ---")
# Total stock count in the South warehouse
print(f"Sum: {df_inventory.wh_south.sum()}")

# Average stock count in the South warehouse
print(f"Mean: {df_inventory.wh_south.mean()}")

# Running cumulative sum down the column
print("\nCumulative Sum:")
print(df_inventory.wh_south.cumsum())
print("\n")

--- Filtered Series: wh_south with data ---
laptops      20.0
firewalls    18.0
Name: wh_south, dtype: float64


--- Drop rows where wh_south is NaN ---
laptops      20.0
firewalls    18.0
Name: wh_south, dtype: float64


--- Drop any row containing at least one NaN ---
Empty DataFrame
Columns: [wh_east, wh_west, wh_north, wh_south, wh_central]
Index: []


--- Drop rows that are completely empty (how='all') ---
           wh_east  wh_west  wh_north  wh_south  wh_central
laptops       15.0     16.0      17.0      20.0         NaN
monitors      18.0     19.0      20.0       NaN         NaN
servers       21.0     22.0      23.0       NaN         NaN
routers       24.0     25.0      26.0       NaN         NaN
switches      27.0     28.0      29.0       NaN         NaN
firewalls     15.0     16.0      17.0      18.0         NaN


--- Drop columns that are completely empty (axis=1) ---
               wh_east  wh_west  wh_north  wh_south
laptops           15.0     16.0      17.0      20.0
mon

In [ ]:
# =====================================================================
# 5. Filling Missing Values
# =====================================================================

# Replace all NaN elements in the system with a flat 0 value
print("--- Fill all NaNs with 0 ---")
filled_inventory = df_inventory.fillna(0)
print(filled_inventory)
print("\n")

# Observe how filling with 0 alters downstream calculations like means
print("--- Mean Comparison: Original vs Filled ---")
print("Original Mean:\n", df_inventory.mean())
print("\nFilled Mean:\n", filled_inventory.mean())
print("\n")

# Forward filling propagates the last valid observation down to next NaN row
print("--- Forward Fill (ffill) on wh_south ---")
print(df_inventory.wh_south.ffill())
print("\n")

# Backward filling propagates the next valid observation up to previous NaN row
print("--- Backward Fill (bfill) on wh_south ---")
print(df_inventory.wh_south.bfill())

--- Fill all NaNs with 0 ---
               wh_east  wh_west  wh_north  wh_south  wh_central
laptops           15.0     16.0      17.0      20.0         0.0
monitors          18.0     19.0      20.0       0.0         0.0
servers           21.0     22.0      23.0       0.0         0.0
routers           24.0     25.0      26.0       0.0         0.0
switches          27.0     28.0      29.0       0.0         0.0
firewalls         15.0     16.0      17.0      18.0         0.0
access_points      0.0      0.0       0.0       0.0         0.0


--- Mean Comparison: Original vs Filled ---
Original Mean:
 wh_east       20.0
wh_west       21.0
wh_north      22.0
wh_south      19.0
wh_central     NaN
dtype: float64

Filled Mean:
 wh_east       17.142857
wh_west       18.000000
wh_north      18.857143
wh_south       5.428571
wh_central     0.000000
dtype: float64


--- Forward Fill (ffill) on wh_south ---
laptops          20.0
monitors         20.0
servers          20.0
routers          20.0
switch

In [ ]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer

# Create sample DataFrame with missing values (NaN) for Tech Hardware Inventory
df_inventory = pd.DataFrame(
    {
        "Device_Age": [2.5, np.nan, 3.0, 4.5, 2.2],
        "Replacement_Cost": [500.00, 600.00, np.nan, 800.00, 450.00],
        "Lifecycle_Status": ["Active", "Phasing_Out", "Active", np.nan, "Phasing_Out"],
    }
)

# 1. Mean Imputation (Numerical)
mean_imputer = SimpleImputer(strategy="mean")
df_inventory["Age_Mean"] = mean_imputer.fit_transform(df_inventory[["Device_Age"]])

# 2. Median Imputation (Numerical - robust to outliers)
median_imputer = SimpleImputer(strategy="median")
df_inventory["Cost_Median"] = median_imputer.fit_transform(df_inventory[["Replacement_Cost"]])

# 3. Mode Imputation (Categorical - most frequent)
mode_imputer = SimpleImputer(strategy="most_frequent")
df_inventory["Status_Mode"] = mode_imputer.fit_transform(df_inventory[["Lifecycle_Status"]]).ravel()

# 4. MICE Imputation (Multivariate Imputation by Chained Equations)
# Select numerical columns for MICE
num_cols = ["Device_Age", "Replacement_Cost"]
mice_imputer = IterativeImputer(max_iter=10, random_state=42)
df_mice = df_inventory[num_cols].copy()
df_mice.iloc[:, :] = mice_imputer.fit_transform(df_mice)
df_inventory["Age_MICE"] = df_mice["Device_Age"]
df_inventory["Cost_MICE"] = df_mice["Replacement_Cost"]

print(df_inventory)


   Device_Age  Replacement_Cost Lifecycle_Status  Age_Mean  Cost_Median  \
0         2.5             500.0           Active      2.50        500.0   
1         NaN             600.0      Phasing_Out      3.05        600.0   
2         3.0               NaN           Active      3.00        550.0   
3         4.5             800.0              NaN      4.50        800.0   
4         2.2             450.0      Phasing_Out      2.20        450.0   

   Status_Mode  Age_MICE   Cost_MICE  
0       Active  2.500000  500.000000  
1  Phasing_Out  3.176643  600.000000  
2       Active  3.000000  573.246038  
3       Active  4.500000  800.000000  
4  Phasing_Out  2.200000  450.000000  
